# Climate comments with full post context / 基于完整帖子语境的评论筛选

**分析单位是一条评论及其帖子语境：帖子标题 + 帖子正文 + 评论正文。** 每条保留记录的 `model_text` 都包含这三部分，并明确标记作者角色。空的帖子正文保留空字段，不虚构内容。

例如帖子讨论碳税，而评论说 “I think this is far too expensive.”，先保留为待复核的语境型评论，确认回应主题后进入主分析；不能仅因原帖相关而自动通过。每条符合基础条件的记录都保存完整语境，包括未被规则选中的记录，便于复核漏选。

这是一套透明的主题候选筛选规则，不是逐条语义判定，也不判断支持或反对立场。相关帖子下仍可能有跑题评论；合并上下文不能保证每条评论都相关，帖子作者的观点也不能当成评论者的观点。

保留原来的至少5词、日期、去重和删除占位符条件。**少于5词的回复仍会被基础条件排除**；`MIN_COMMENT_WORDS` 可以调整。这里的上下文是原帖，不含父评论链。

原始数据库不变。本次编辑前的08和输出备份在 `outputs/08_before_combined_context_20261001`。输出继续使用 `distillation_v2_output` 中的兼容文件名，通过 `selection_logic_version` 标记新规则。

## 1. Configuration and input / 配置与输入

In [1]:
import os
import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", None)
SOURCE_PATH = Path(os.environ.get("DM1_SOURCE_PATH", r"D:\DM1_project\reddit_opinion_climate_change_cleaned_final.parquet"))
OUTPUT_DIR = Path(os.environ.get("DM1_OUTPUT_DIR", r"D:\DM1_project\distillation_v2_output"))
DEV_AUDIT_PATH = Path(os.environ.get("DM1_DEV_AUDIT", r"D:\DM1_project\DM1_distillation_assistant_reviewed_500.csv"))
REVIEW_LABELS_PATH = Path(os.environ.get("DM1_REVIEW_LABELS", str(OUTPUT_DIR / "context_review_labels.csv")))
START_DATE = pd.Timestamp("2023-11-19")
END_EXCLUSIVE = pd.Timestamp("2026-09-01")
MIN_COMMENT_WORDS = 5
BATCH_SIZE = 50_000
RANDOM_STATE = 42
LOGIC_VERSION = "combined_context_tiered_review_v1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert SOURCE_PATH.exists(), SOURCE_PATH
source_file = pq.ParquetFile(SOURCE_PATH)
required = {"comment_id", "post_id", "subreddit", "created_time", "post_title", "post_self_text", "clean_text"}
assert required.issubset(source_file.schema_arrow.names), required - set(source_file.schema_arrow.names)
print("Input rows:", f"{source_file.metadata.num_rows:,}")
print("Matching unit: POST TITLE + POST BODY + COMMENT")
print("Minimum comment words:", MIN_COMMENT_WORDS)
review_labels = pd.DataFrame(columns=["comment_id", "review_relevance", "review_reason", "review_stance", "stance_target"])
if REVIEW_LABELS_PATH.exists():
    review_labels = pd.read_csv(REVIEW_LABELS_PATH, dtype=str).fillna("")
    assert {"comment_id", "review_relevance"}.issubset(review_labels.columns)
    assert review_labels["comment_id"].ne("").all() and review_labels["comment_id"].is_unique
    review_labels["review_relevance"] = review_labels["review_relevance"].str.strip().str.lower()
    assert review_labels["review_relevance"].isin(["", "relevant", "irrelevant", "uncertain"]).all()
review_map = review_labels.set_index("comment_id")["review_relevance"].to_dict()
print("Completed relevance reviews:", sum(value != "" for value in review_map.values()))


Input rows: 1,894,876
Matching unit: POST TITLE + POST BODY + COMMENT
Minimum comment words: 5
Completed relevance reviews: 250


## 2. Vocabulary / 主题词规则

沿用旧08的明确主题词、相邻主题词及歧义表达，便于比较。明确词包含气候、减排、能源等研究范围，不只包括字面上的“climate change”。

变化发生在上下文的使用方式：不再禁止 `changemyview`、`conspiracy` 进入语境候选层，不再要求帖子正文至少命中两个词。评论缺少支持词时仍保存为待复核，不能自动进入主样本。社区名称本身不构成保留依据。

歧义 `climate` 表达先移除，再检查剩余文字。无关语境标记用于人工复核提示，不对同时出现明确气候主题的整段文字一刀切删除。

In [2]:
CLIMATE_FOCUSED_SUBREDDITS = {
    "climate", "climatechange", "globalclimatechange",
    "climateactionplan", "climateoffensive", "citizensclimatelobby",
    "climateco", "climateskeptics", "climate_nuremberg",
}

NOISY_GENERAL_SUBREDDITS = {"changemyview", "conspiracy"}

EXPLICIT_PATTERN = r"""
\b(?:
    climate\s+(?:change|crisis|policy|policies|action|science|scientist|scientists|
        denial|denier|deniers|adaptation|mitigation|justice|emergency|warming|hoax|
        skeptic|skeptics|sceptic|sceptics|commitment|commitments|target|targets|goal|goals|
        protest|protests|protester|protesters|advocate|advocates|grant|grants|choice|choices|
        risk|risks|impact|impacts|anxiety|sensitivity|feedback|fact[-\s]?check|finance|
        diplomacy|voter|voters|job|jobs|career|careers|solution|solutions|technology|tech|
        research|report|reports|model|models|data|disaster|resilience|resilient|related)|
    climatic\s+change|climatolog(?:y|ist|ists)|global\s+(?:warming|heating)|
    greenhouse\s+(?:gas|gases|effect|emission|emissions)|
    carbon\s+(?:dioxide|emission|emissions|tax|taxes|capture|budget|neutrality|footprint|
        credit|credits|pricing)|co2|methane\s+emissions?|net\s+zero|decarbon\w*|
    fossil\s+fuels?|big\s+oil|paris\s+(?:agreement|accord)|ipcc|
    renewable\s+(?:energy|power|electricity)|clean\s+energy|green\s+energy|
    solar\s+(?:energy|power|panel|panels)|photovoltaic\w*|
    wind\s+(?:energy|power|turbine|turbines)|
    coal\s+(?:power|plant|plants|industry|emission|emissions)|
    oil\s+(?:industry|industries|companies|company|production|drilling)|
    natural\s+gas|energy\s+(?:transition|policy|policies|crisis|system|systems|market|markets|
        grid|security|independence)|electric\s+vehicles?|\bevs?\b|ice\s+vehicles?|
    zero[-\s]emission\s+vehicles?|nuclear\s+(?:power|energy|reactor|reactors|plant|plants|grid)|
    small\s+modular\s+reactors?|sea\s+level\s+(?:rise|rises|rising)|
    melting\s+(?:ice|glaciers?)|thawing\s+permafrost|arctic\s+sea\s+ice|
    (?:cut|cuts|cutting|reduce|reduces|reduced|reducing|limit|limits|limited|limiting|
        lower|lowers|lowered|lowering)\s+(?:carbon\s+)?emissions?|
    emissions?\s+(?:target|targets|cut|cuts|reduction|reductions)|
    temperature\s+(?:rise|rises|rising)|rising\s+temperatures?|hotter\s+winters?|
    save\s+(?:the\s+)?planet
)\b
"""

ADJACENT_PATTERN = r"""
\b(?:
    record[-\s](?:breaking\s+)?(?:heat|temperature|temperatures|warmth)|
    hottest\s+(?:day|month|year|temperature|temperatures)|
    heat\s*waves?|droughts?|wildfires?|glaciers?|ice\s+sheets?|permafrost|
    extreme\s+(?:weather|heat|rainfall|flooding)|
    environmental\s+(?:policy|policies|impact|impacts|protection|damage|crisis)|
    sustainability|sustainable\s+energy|energy\s+storage|battery\s+storage|
    carbon[-\s]intensive|low[-\s]carbon
)\b
"""

COMMENT_SUPPORT_PATTERN = r"""
\b(?:
    emissions?|pollution|warming|heating|hotter|temperature|temperatures|weather|
    methane|carbon|co2|fossil|oil|coal|gas|renewable|solar|photovoltaic|wind|
    nuclear|reactor|reactors|uranium|electricity|power\s+(?:plant|plants|grid|generation)|
    grid|battery|batteries|drought|droughts|flood|flooding|wildfire|wildfires|
    glacier|glaciers|ice|permafrost|ocean|environment|environmental|ecology|ecological|
    planet|sustainability|sustainable|mitigation|adaptation|resilience|greenwashing|
    conservation|deforestation|sea\s+level|heat\s*wave|heatwaves
)\b
"""

OFF_TOPIC_CONTEXT_PATTERN = r"""
\b(?:numerolog\w*|chakra\w*|ley\s+lines?|astrolog\w*|antichrist|telepathy|
    alien\s+invasion|flat\s+earth|moon\s+landing|diddy\s+part(?:y|ies)|
    covid\s+variant|trans\s+gendre|rothschild)\b|\b666\b|
(?:\b(?:nuclear|solar|energy)\b.{0,40}\b(?:signal|signalling|god|spiritual)\b)
"""

NEGATED_CLIMATE_PATTERN = r"""
\b(?:political|economic|business|social|cultural|investment|regulatory|workplace|current|cold)
    \s+climate\b|\bclimate\s+of\s+(?:fear|opinion|hate|violence|uncertainty)\b
"""

DIRECT_OFF_TOPIC_PATTERN = r"""
\bnet\s+zero\s+body\s+count\b|
\b(?:venus|mars|martian|lunar)\b.{0,80}\b(?:carbon\s+dioxide|co2|terraform\w*)\b|
\b(?:north\s+korea|nuclear\s+weapon|nuclear\s+weapons|warhead|warheads|missile|missiles|
    nuclear\s+deterrent)\b
"""

FLAGS = re.IGNORECASE | re.VERBOSE
EXPLICIT_RE = re.compile(EXPLICIT_PATTERN, FLAGS)
ADJACENT_RE = re.compile(ADJACENT_PATTERN, FLAGS)
COMMENT_SUPPORT_RE = re.compile(COMMENT_SUPPORT_PATTERN, FLAGS)
OFF_TOPIC_CONTEXT_RE = re.compile(OFF_TOPIC_CONTEXT_PATTERN, FLAGS)
NEGATED_CLIMATE_RE = re.compile(NEGATED_CLIMATE_PATTERN, FLAGS)
DIRECT_OFF_TOPIC_RE = re.compile(DIRECT_OFF_TOPIC_PATTERN, FLAGS)
BARE_CLIMATE_RE = re.compile(r"\bclimate\b", re.IGNORECASE)
EXPLICIT_EXTRACT_RE = re.compile(f"({EXPLICIT_PATTERN})", FLAGS)

print("Revised patterns compiled successfully.")

Revised patterns compiled successfully.


## 3. One shared scoring function / 统一评分逻辑

层级记录相关性证据来自哪里。关键词证据仍是初筛，不是已确认的语义相关性：

| 层级 | 条件 | 主样本 |
|---|---|---|
| `direct_comment` | 评论自身命中明确主题词或非已知歧义的 climate | 暂纳入；有可疑标记则复核 |
| `supported_context` | 原帖命中，评论也有能源环境等支持词 | 暂纳入；有可疑标记则复核 |
| `context_only` | 原帖命中，评论无需重复主题词 | **确认相关后纳入** |
| `adjacent_review` | 只有相邻主题或可疑的弱 climate 线索 | 待复核 |
| `not_selected` | 三部分均无足够主题线索 | 否；不等同于语义无关 |

人工复核优先：`relevant` 可将任意层提升到主样本；`irrelevant` 排除；`uncertain` 留在待复核。`review_relevance` 判断是否讨论研究主题，`review_stance` 和 `stance_target` 另行记录立场及其对象；有主题相关性不代表立场明确。

为避免反复匹配同一个原帖，先分别计算字段证据。模型使用的文本始终包含三部分；不会把不同字段尾首的词误拼成一个关键词。

In [3]:
POST_STRINGS = ["matched_title_term_v2", "matched_post_body_term_v2"]
POST_BOOLS = ["title_explicit_match", "body_explicit_match", "title_adjacent_match", "body_adjacent_match",
              "title_off_topic", "body_off_topic", "post_direct_off_topic", "title_bare_climate",
              "body_bare_climate", "strong_title_context", "strong_body_context", "post_context_allowed"]

def text_column(frame, name):
    return frame[name].fillna("").astype(str).str.strip()

def field_evidence(text):
    disambiguated = text.str.replace(NEGATED_CLIMATE_RE, " ", regex=True)
    disambiguated = disambiguated.str.replace(r"(?i)\bclimate\s+(?:control|controlled)\b", " ", regex=True)
    explicit = disambiguated.str.contains(EXPLICIT_RE, na=False)
    bare = disambiguated.str.contains(BARE_CLIMATE_RE, na=False)
    off_topic = text.str.contains(OFF_TOPIC_CONTEXT_RE, na=False) | text.str.contains(DIRECT_OFF_TOPIC_RE, na=False)
    strong = explicit | (bare & ~off_topic)
    return {
        "explicit": explicit, "bare": bare, "off": off_topic, "strong": strong,
        "adjacent": disambiguated.str.contains(ADJACENT_RE, na=False),
        "term": disambiguated.str.extract(EXPLICIT_EXTRACT_RE, expand=False).fillna("").where(~(bare & ~explicit), "climate"),
    }

def score_posts(frame):
    posts = frame[["post_id", "subreddit", "post_title", "post_self_text"]].drop_duplicates("post_id").copy()
    title = field_evidence(text_column(posts, "post_title"))
    body = field_evidence(text_column(posts, "post_self_text"))
    for prefix, evidence in [("title", title), ("body", body)]:
        posts[prefix + "_explicit_match"] = evidence["explicit"]
        posts[prefix + "_adjacent_match"] = evidence["adjacent"]
        posts[prefix + "_off_topic"] = evidence["off"]
        posts[prefix + "_bare_climate"] = evidence["bare"]
        posts["strong_" + prefix + "_context"] = evidence["strong"]
    posts["post_direct_off_topic"] = title["off"] | body["off"]
    posts["post_context_allowed"] = title["strong"] | body["strong"]
    posts["matched_title_term_v2"] = title["term"]
    posts["matched_post_body_term_v2"] = body["term"]
    return posts

def score_records(frame, post_scores=None, labels=None):
    if post_scores is None:
        post_scores = score_posts(frame)
    flags = post_scores[["post_id"] + POST_BOOLS + POST_STRINGS]
    scored = frame.merge(flags, on="post_id", how="left", validate="many_to_one")
    assert scored["post_context_allowed"].notna().all()
    comment = text_column(scored, "clean_text")
    evidence = field_evidence(comment)
    scored["comment_explicit_match_v2"] = evidence["explicit"]
    scored["comment_support_match_v2"] = comment.str.contains(COMMENT_SUPPORT_RE, na=False)
    scored["comment_adjacent_match_v2"] = evidence["adjacent"]
    scored["comment_direct_off_topic"] = evidence["off"]
    scored["direct_comment"] = evidence["strong"]
    scored["supported_context"] = ~scored["direct_comment"] & scored["post_context_allowed"] & scored["comment_support_match_v2"]
    scored["context_only"] = ~scored["direct_comment"] & scored["post_context_allowed"] & ~scored["supported_context"]
    included = scored["direct_comment"] | scored["post_context_allowed"]
    weak = (scored["title_adjacent_match"] | scored["body_adjacent_match"] | evidence["adjacent"]
            | scored["title_bare_climate"] | scored["body_bare_climate"] | evidence["bare"])
    scored["adjacent_review"] = ~included & weak
    scored["selection_tier_v2"] = np.select(
        [scored["direct_comment"], scored["supported_context"], scored["context_only"], scored["adjacent_review"]],
        ["direct_comment", "supported_context", "context_only", "adjacent_review"], default="not_selected")
    scored["needs_manual_review_v2"] = scored["context_only"] | scored["adjacent_review"] | evidence["off"] | scored["title_off_topic"] | scored["body_off_topic"]
    decisions = review_map if labels is None else labels
    scored["review_relevance"] = scored["comment_id"].astype(str).map(decisions).fillna("")
    confirmed = scored["review_relevance"].eq("relevant")
    rejected = scored["review_relevance"].eq("irrelevant")
    uncertain = scored["review_relevance"].eq("uncertain")
    reviewed = scored["review_relevance"].ne("")
    automatic_main = (scored["direct_comment"] | scored["supported_context"]) & ~scored["needs_manual_review_v2"]
    scored["cluster_ready_v2"] = confirmed | (automatic_main & ~reviewed)
    scored["context_analysis_v2"] = (included | confirmed) & ~rejected
    scored["pending_review_v2"] = uncertain | (scored["needs_manual_review_v2"] & ~reviewed)
    scored["main_selection_reason"] = np.select(
        [confirmed, rejected, scored["pending_review_v2"], scored["cluster_ready_v2"]],
        ["confirmed_by_review", "excluded_by_review", "awaiting_review", "provisional_rule_evidence"],
        default="insufficient_rule_evidence")
    scored["matched_comment_term_v2"] = evidence["term"]
    scored["selection_logic_version"] = LOGIC_VERSION
    scored["model_text"] = (
        "[POST TITLE] " + text_column(scored, "post_title")
        + "\n[POST BODY] " + text_column(scored, "post_self_text")
        + "\n[COMMENT] " + comment
    )
    return scored


## 4. Verify context-dependent replies / 验证依赖上下文的回复

In [4]:
examples = pd.DataFrame([
    ["a", "climatechange", "Should we adopt a carbon tax?", "Would it reduce emissions?", "I think this is far too expensive."],
    ["b", "climatechange", "Should we adopt a carbon tax?", "Would it reduce emissions?", "I think electricity would become too expensive."],
    ["c", "climatechange", "Should we adopt a carbon tax?", "Would it reduce emissions?", "I oppose this carbon tax because of its cost."],
    ["d", "changemyview", "What do you think of this proposal?", "The proposal is a carbon tax.", "I think this is far too expensive."],
    ["e", "climatechange", "Is this laptop worth buying?", "I need a laptop for gaming.", "I think this is far too expensive."],
    ["f", "changemyview", "The political climate is changing", "This is about election tactics.", "I disagree with that proposed strategy."],
    ["g", "climatechange", "A question for this community", "", "I do not believe that climate change is real."],
    ["h", "conspiracy", "Any thoughts?", "Global warming is mentioned here.", "I disagree with the proposed response."],
], columns=["post_id", "subreddit", "post_title", "post_self_text", "clean_text"])
examples["comment_id"] = "test_" + examples["post_id"]
checked = score_records(examples, labels={})
assert checked["selection_tier_v2"].tolist() == ["context_only", "supported_context", "direct_comment", "context_only", "not_selected", "not_selected", "direct_comment", "context_only"]
assert checked["cluster_ready_v2"].tolist() == [False, True, True, False, False, False, True, False]
assert checked.loc[3, "model_text"] == "[POST TITLE] What do you think of this proposal?\n[POST BODY] The proposal is a carbon tax.\n[COMMENT] I think this is far too expensive."
reviewed_examples = score_records(examples, labels={"test_a": "relevant", "test_b": "irrelevant", "test_c": "uncertain"})
assert bool(reviewed_examples.loc[0, "cluster_ready_v2"])
assert not bool(reviewed_examples.loc[1, "cluster_ready_v2"])
assert not bool(reviewed_examples.loc[1, "context_analysis_v2"])
assert not bool(reviewed_examples.loc[2, "cluster_ready_v2"])
assert bool(reviewed_examples.loc[2, "pending_review_v2"])
display(checked[["post_title", "post_self_text", "clean_text", "selection_tier_v2", "cluster_ready_v2"]])
print("Passed: context-only review gate, review promotion/rejection/deferral, body context, unrelated controls, and climate denial.")


,post_title,post_self_text,clean_text,selection_tier_v2,cluster_ready_v2
0,Should we adopt a carbon tax?,Would it reduce emissions?,I think this is far too expensive.,context_only,False
1,Should we adopt a carbon tax?,Would it reduce emissions?,I think electricity would become too expensive.,supported_context,True
2,Should we adopt a carbon tax?,Would it reduce emissions?,I oppose this carbon tax because of its cost.,direct_comment,True
3,What do you think of this proposal?,The proposal is a carbon tax.,I think this is far too expensive.,context_only,False
4,Is this laptop worth buying?,I need a laptop for gaming.,I think this is far too expensive.,not_selected,False
5,The political climate is changing,This is about election tactics.,I disagree with that proposed strategy.,not_selected,False
6,A question for this community,,I do not believe that climate change is real.,direct_comment,True
7,Any thoughts?,Global warming is mentioned here.,I disagree with the proposed response.,context_only,False


Passed: context-only review gate, review promotion/rejection/deferral, body context, unrelated controls, and climate denial.


## 5. Score posts and stream comments / 计算原帖证据并逐批处理评论

保留原始字段，添加完整语境和证据字段。分批处理避免一次性复制所有长文本。按原文件顺序保留第一个评论ID；剔除缺失ID、空白/删除占位符、少于5词和时间范围外的记录。

输出分为全部候选、主样本、语境扩展样本、待复核记录和相邻主题记录。主样本不会自动包含仅靠原帖相关的评论。处理和行数核查成功后才替换正式输出。

In [5]:
post_input = pd.read_parquet(SOURCE_PATH, columns=["post_id", "subreddit", "post_title", "post_self_text"])
post_input = post_input.dropna(subset=["post_id"]).drop_duplicates("post_id", keep="first")
post_scores = score_posts(post_input)
del post_input
print("Posts scored:", f"{len(post_scores):,}", flush=True)
display(post_scores[["strong_title_context", "strong_body_context", "post_context_allowed"]].sum().to_frame("posts"))


Posts scored: 61,494


,posts
strong_title_context,33442
strong_body_context,9519
post_context_allowed,39081


In [6]:
STRING_COLUMNS = POST_STRINGS + ["matched_comment_term_v2", "selection_tier_v2", "selection_logic_version", "model_text", "review_relevance", "main_selection_reason"]
BOOL_COLUMNS = POST_BOOLS + ["comment_explicit_match_v2", "comment_support_match_v2", "comment_adjacent_match_v2", "comment_direct_off_topic",
    "direct_comment", "supported_context", "context_only", "adjacent_review", "cluster_ready_v2", "context_analysis_v2", "needs_manual_review_v2", "pending_review_v2"]
schema_fields = list(source_file.schema_arrow)
if "comment_word_count" not in source_file.schema_arrow.names:
    schema_fields.append(pa.field("comment_word_count", pa.int64()))
output_schema = pa.schema(schema_fields + [pa.field(c, pa.bool_()) for c in BOOL_COLUMNS] + [pa.field(c, pa.string()) for c in STRING_COLUMNS])
paths = {
    "all": OUTPUT_DIR / "reddit_climate_candidates_v2.parquet",
    "cluster": OUTPUT_DIR / "reddit_climate_cluster_ready_v2.parquet",
    "context": OUTPUT_DIR / "reddit_climate_distilled_v2.parquet",
    "review": OUTPUT_DIR / "reddit_climate_adjacent_review_v2.parquet",
    "pending": OUTPUT_DIR / "reddit_climate_context_review_v2.parquet",
}
temp_paths = {key: path.with_suffix(".pending.parquet") for key, path in paths.items()}
writers = {key: pq.ParquetWriter(temp_paths[key], output_schema, compression="snappy") for key in paths}
seen_ids = set()
selected_posts = set()
tier_counts = Counter()
community_tier_counts = Counter()
stage_counts = Counter()
review_flag_count = 0
output_counts = Counter()
main_tier_counts = Counter()
community_main_counts = Counter()
reviewed_eligible_ids = set()
sample_pools = {}
rng = np.random.default_rng(RANDOM_STATE)
SAMPLE_SIZE = 50
processed = 0

try:
    for batch in source_file.iter_batches(batch_size=BATCH_SIZE):
        frame = batch.to_pandas()
        stage_counts["Input records"] += len(frame)
        frame = frame.dropna(subset=["comment_id", "post_id", "subreddit"])
        stage_counts["Require identifiers"] += len(frame)
        frame = frame.drop_duplicates("comment_id", keep="first")
        frame = frame.loc[~frame["comment_id"].isin(seen_ids)].copy()
        seen_ids.update(frame["comment_id"])
        stage_counts["Unique comment IDs"] += len(frame)
        text = text_column(frame, "clean_text")
        frame = frame.loc[~text.str.lower().isin({"", "[deleted]", "[removed]", "deleted", "removed"})].copy()
        stage_counts["Nonempty comment text"] += len(frame)
        if "comment_word_count" not in frame:
            frame["comment_word_count"] = text_column(frame, "clean_text").str.split().str.len()
        frame = frame.loc[frame["comment_word_count"].ge(MIN_COMMENT_WORDS)].copy()
        stage_counts["Minimum comment length"] += len(frame)
        dates = pd.to_datetime(frame["created_time"], errors="coerce")
        frame = frame.loc[dates.ge(START_DATE) & dates.lt(END_EXCLUSIVE)].copy()
        stage_counts["Study date window"] += len(frame)
        processed += batch.num_rows
        if frame.empty:
            continue
        scored = score_records(frame, post_scores)
        assert not (scored["context_only"] & scored["cluster_ready_v2"] & scored["review_relevance"].ne("relevant")).any()
        assert not (scored["review_relevance"].isin(["irrelevant", "uncertain"]) & scored["cluster_ready_v2"]).any()
        tier_counts.update(scored["selection_tier_v2"])
        main_tier_counts.update(scored.loc[scored["cluster_ready_v2"], "selection_tier_v2"])
        community_main_counts.update(scored.loc[scored["cluster_ready_v2"], "subreddit"])
        reviewed_eligible_ids.update(scored.loc[scored["review_relevance"].ne(""), "comment_id"])
        community_tier_counts.update(zip(scored["subreddit"], scored["selection_tier_v2"]))
        review_flag_count += int((scored["cluster_ready_v2"] & scored["needs_manual_review_v2"]).sum())
        selected_posts.update(scored.loc[scored["cluster_ready_v2"], "post_id"])
        for col in STRING_COLUMNS:
            scored[col] = scored[col].astype("string")
        selected = scored.loc[scored["cluster_ready_v2"]]
        subsets = {"all": scored, "cluster": selected, "context": scored.loc[scored["context_analysis_v2"]],
                   "review": scored.loc[scored["adjacent_review"]], "pending": scored.loc[scored["pending_review_v2"]]}
        for key, subset in subsets.items():
            writers[key].write_table(pa.Table.from_pandas(subset, schema=output_schema, preserve_index=False))
            output_counts[key] += len(subset)
        # Uniform random-key samples across batches; no preference for popular posts.
        sample_columns = ["comment_id", "post_id", "subreddit", "post_title", "post_self_text", "clean_text", "model_text", "selection_tier_v2", "needs_manual_review_v2", "cluster_ready_v2", "review_relevance"]
        candidates = scored[sample_columns].assign(_random_key=rng.random(len(scored)))
        for tier, group in candidates.groupby("selection_tier_v2", sort=False):
            pool = pd.concat([sample_pools.get(tier, group.iloc[:0]), group.nsmallest(SAMPLE_SIZE, "_random_key")], ignore_index=True)
            sample_pools[tier] = pool.nsmallest(SAMPLE_SIZE, "_random_key")
        if processed % 100_000 == 0 or processed == source_file.metadata.num_rows:
            print(f"Processed {processed:,}/{source_file.metadata.num_rows:,}; eligible {stage_counts['Study date window']:,}; main {output_counts['cluster']:,}; pending {output_counts['pending']:,}", flush=True)
finally:
    for writer in writers.values():
        writer.close()

eligible_count = stage_counts["Study date window"]
selected_count = output_counts["cluster"]
assert sum(tier_counts.values()) == eligible_count
assert output_counts["all"] == eligible_count
for key in paths:
    assert pq.ParquetFile(temp_paths[key]).metadata.num_rows == output_counts[key]
for key in paths:
    temp_paths[key].replace(paths[key])
print("All output row-count checks passed.")


Processed 100,000/1,894,876; eligible 69,778; main 17,853; pending 33,639


Processed 200,000/1,894,876; eligible 162,886; main 39,710; pending 79,398


Processed 300,000/1,894,876; eligible 255,492; main 62,968; pending 115,267


Processed 400,000/1,894,876; eligible 348,959; main 86,281; pending 151,956


Processed 500,000/1,894,876; eligible 442,342; main 111,080; pending 189,801


...

Processed 1,500,000/1,894,876; eligible 1,370,684; main 356,470; pending 582,755


Processed 1,600,000/1,894,876; eligible 1,463,357; main 381,019; pending 621,446


Processed 1,700,000/1,894,876; eligible 1,556,431; main 410,133; pending 659,909


Processed 1,800,000/1,894,876; eligible 1,642,629; main 437,843; pending 695,836


All output row-count checks passed.


## 6. Results and saved evidence / 统计与证据输出

In [7]:
TIER_ORDER = ["direct_comment", "supported_context", "context_only", "adjacent_review", "not_selected"]
tier_summary = pd.DataFrame({"selection_tier_v2": TIER_ORDER, "comments": [tier_counts[t] for t in TIER_ORDER]})
tier_summary["percent_of_eligible"] = (tier_summary["comments"] / eligible_count * 100).round(2)
tier_summary["main_sample_comments"] = [main_tier_counts[t] for t in TIER_ORDER]
display(tier_summary)
community_summary = pd.DataFrame([
    {"subreddit": community, "selection_tier_v2": tier, "comments": count}
    for (community, tier), count in community_tier_counts.items()
]).pivot(index="subreddit", columns="selection_tier_v2", values="comments").fillna(0).astype(int).reindex(columns=TIER_ORDER, fill_value=0)
community_summary["main_sample"] = community_summary.index.map(lambda key: community_main_counts[key])
community_summary["eligible"] = community_summary[TIER_ORDER].sum(axis=1)
display(community_summary.sort_values("eligible", ascending=False))

funnel_rows = []
before = source_file.metadata.num_rows
for step, after in stage_counts.items():
    funnel_rows.append({"section": "cleaning_funnel", "step": step, "rows_before": before, "rows_after": after,
                        "rows_removed": before-after, "retained_percent": round(after/before*100, 2) if before else np.nan})
    before = after
audit = pd.concat([pd.DataFrame(funnel_rows), tier_summary.rename(columns={"selection_tier_v2": "step", "comments": "rows_after"}).assign(section="selection_tier")], ignore_index=True)
audit["selection_logic_version"] = LOGIC_VERSION
audit.to_csv(OUTPUT_DIR / "reddit_climate_distillation_v2_audit.csv", index=False, encoding="utf-8-sig")
community_summary.to_csv(OUTPUT_DIR / "reddit_climate_distillation_v2_by_community.csv", encoding="utf-8-sig")
retained_posts = post_scores.loc[post_scores["post_id"].isin(selected_posts)].copy()
retained_posts.to_parquet(OUTPUT_DIR / "reddit_climate_distilled_posts_v2.parquet", index=False)
review_samples = pd.concat(sample_pools.values(), ignore_index=True).drop(columns="_random_key")
review_samples.to_parquet(OUTPUT_DIR / "reddit_climate_context_review_samples_v2.parquet", index=False)
if not REVIEW_LABELS_PATH.exists():
    template = review_samples.copy()
    template["review_reason"] = ""
    template["review_stance"] = ""
    template["stance_target"] = ""
    template.to_csv(REVIEW_LABELS_PATH, index=False, encoding="utf-8-sig")
    print("Created review form:", REVIEW_LABELS_PATH)
else:
    print("Existing review form preserved:", REVIEW_LABELS_PATH)
print("Applied completed reviews:", len(reviewed_eligible_ids))
print("Review IDs outside eligible input:", len({key for key, value in review_map.items() if value} - reviewed_eligible_ids))
print(f"Eligible comments: {eligible_count:,}; main sample: {selected_count:,}; context-only main sample: {main_tier_counts['context_only']:,}")
print("Output row counts:", dict(output_counts))
print(f"Selected comments with review flags: {review_flag_count:,}; retained posts: {len(retained_posts):,}")
print("Each retained model_text includes post title, post body, and comment.")


,selection_tier_v2,comments,percent_of_eligible,main_sample_comments
0,direct_comment,291858,17.77,289380
1,supported_context,150882,9.19,148420
2,context_only,559400,34.06,19
3,adjacent_review,89739,5.46,21
4,not_selected,550750,33.53,3


selection_tier_v2,direct_comment,supported_context,context_only,adjacent_review,not_selected,main_sample,eligible
subreddit,,,,,,,
conspiracy,8649,4636,32930,6073,161302,12474,213590
climatechange,64808,27779,78384,6650,12916,91980,190537
changemyview,10657,7323,54814,3078,105654,16798,181526
climate,40121,16959,74107,7261,13445,56894,151893
Futurology,18729,13388,44976,1603,33654,31313,112350
energy,27118,20336,53960,1168,5466,47388,108048
europe,16465,10235,23904,14056,33945,26534,98605
worldnews,13857,6917,25042,12664,37683,20504,96163
politics,7706,3712,24834,1382,41004,11329,78638


Existing review form preserved: D:\DM1_project\distillation_v2_output\context_review_labels.csv
Applied completed reviews: 250
Review IDs outside eligible input: 0
Eligible comments: 1,642,629; main sample: 437,843; context-only main sample: 19
Output row counts: {'all': 1642629, 'cluster': 437843, 'context': 1002129, 'review': 89739, 'pending': 695836}
Selected comments with review flags: 40; retained posts: 35,626
Each retained model_text includes post title, post body, and comment.


## 7. Inspect context-dependent replies / 抽查完整语境

下方展示随机抽样中的前10条仅依赖语境评论。完整分层样本另存为 `reddit_climate_context_review_samples_v2.parquet`，每层最多50条。抽查已纳入和未纳入两侧，才能同时发现误收与漏收。各层等量抽样不能直接按总样本平均来估计全体相关率。

首次运行生成 `context_review_labels.csv`，后续不会覆盖已填内容。填写 `review_relevance` 为 `relevant` / `irrelevant` / `uncertain`，并写下 `review_reason`；重新运行08即可应用。相关但立场不清晰时仍可标为 relevant，另在 `review_stance` 标为 unclear；表达支持/反对时记录 `stance_target`。如果缺少父评论导致无法判断，标 uncertain，不强行分类。

本 notebook 不自动填写人工判断，不报告未经验证的准确率，保留原来的100条用户审核样本不变。

In [8]:
context_examples = review_samples.loc[review_samples["selection_tier_v2"].eq("context_only"), ["comment_id", "subreddit", "model_text"]].head(10)
display(context_examples.style.set_properties(**{"white-space": "pre-wrap", "text-align": "left", "vertical-align": "top"})
        .set_properties(subset=["model_text"], **{"min-width": "750px", "max-width": "1100px"}).hide(axis="index"))


## 8. Optional development validation / 可选开发集验证

仅当原来的500条开发审核文件存在时执行。验证调用同一个评分函数；如果只有截断的 preview 字段，结果只代表该截断文本上的表现。不会把已有100条用户样本当作调参开发集。

In [9]:
validation_summary = pd.DataFrame()
if DEV_AUDIT_PATH.exists():
    dev = pd.read_csv(DEV_AUDIT_PATH)
    validation_input = pd.DataFrame({
        "comment_id": "dev_" + pd.Series(np.arange(len(dev)).astype(str)),
        "post_id": np.arange(len(dev)).astype(str),
        "subreddit": dev["subreddit"],
        "post_title": dev["post_title"],
        "post_self_text": dev["post_self_text"] if "post_self_text" in dev else dev["post_body_preview"],
        "clean_text": dev["clean_text"] if "clean_text" in dev else dev["comment_preview"],
    })
    scored_dev = score_records(validation_input, labels={})
    expected = dev["review_comment_relevance"].isin(["direct", "contextual"])
    selected = scored_dev["cluster_ready_v2"]
    tp, fp, fn = int((selected & expected).sum()), int((selected & ~expected).sum()), int((~selected & expected).sum())
    validation_summary = pd.DataFrame([{"true_positive": tp, "false_positive": fp, "false_negative": fn,
        "precision": tp/(tp+fp) if tp+fp else np.nan, "recall": tp/(tp+fn) if tp+fn else np.nan}])
    display(validation_summary)
else:
    print("Development audit not found; empirical precision/recall not evaluated.")


Development audit not found; empirical precision/recall not evaluated.


## 9. Downstream use / 后续分析要求

主分析读取 `reddit_climate_cluster_ready_v2.parquet`，**直接使用已经保存的 `model_text`，以 `cluster_ready_v2` 为准**。`selection_tier_v2` 只表示规则证据来源，不能覆盖人工复核后的纳入决定。仅依赖原帖的评论必须确认相关后才进入主样本。

`reddit_climate_distilled_v2.parquet` 是更宽的语境扩展样本，可比较纳入未确认语境评论前后的结果；已人工标为 irrelevant 的评论不进入扩展样本。`reddit_climate_candidates_v2.parquet` 保存全部基础合格记录及完整语境。`reddit_climate_context_review_v2.parquet` 保存待复核记录，包括有可疑语境的主规则命中项。

09已更新为读取 `cluster_ready_v2` 和08保存的完整 `model_text`，并使用内容指纹创建新向量缓存。**09尚未重新运行**；它之前生成的聚类文件仍是旧结果，必须重新运行09才能得到与本次语料一致的结果。

完整 `model_text` 未截断。下游模型存在长度限制时，需要明确处理长原帖，保证评论正文不会被末尾截断；本 notebook 不执行模型推理或向量生成。

关键词与原帖语境筛选扩大了候选覆盖范围，但仍须通过完整语境抽样审核来估计跑题率和立场可判定率。

## 10. Review results by tier / 按层级汇总人工复核

抽样表中每层随机50条。先复核 `context_only`、`direct_comment`、`supported_context`，分别衡量可能漏掉的有效回复和主样本的误收。随后复核 `not_selected` 与 `adjacent_review`，检查漏选。

只有每层至少20条得到明确的相关/无关判断时才计算该层样本相关率。`uncertain` 单独计数，不强行当作相关或无关。各层样本数量相同，但总体大小不同，因此不能把各层比例直接平均；扩展决策应同时考虑该层总体规模。这里的比例仅是随机样本的粗略估计，不能自动批准整层进入主样本。

In [10]:
review_sheet = pd.read_csv(REVIEW_LABELS_PATH, dtype=str, keep_default_na=False)
review_sheet["review_relevance"] = review_sheet["review_relevance"].str.strip().str.lower()
review_rows = []
for tier in TIER_ORDER:
    group = review_sheet.loc[review_sheet["selection_tier_v2"].eq(tier)]
    relevant = int(group["review_relevance"].eq("relevant").sum())
    irrelevant = int(group["review_relevance"].eq("irrelevant").sum())
    uncertain = int(group["review_relevance"].eq("uncertain").sum())
    known = relevant + irrelevant
    review_rows.append({
        "tier": tier,
        "population_comments": tier_counts[tier],
        "sample_comments": len(group),
        "reviewed_relevant": relevant,
        "reviewed_irrelevant": irrelevant,
        "reviewed_uncertain": uncertain,
        "not_yet_reviewed": len(group) - known - uncertain,
        "sample_relevant_percent_if_n_ge_20": round(100 * relevant / known, 1) if known >= 20 else np.nan,
    })
review_progress = pd.DataFrame(review_rows)
display(review_progress)
if (review_progress["reviewed_relevant"] + review_progress["reviewed_irrelevant"]).sum() == 0:
    print("No completed relevance reviews yet. Label the sample CSV, then rerun Notebook 08.")
else:
    print("Sample rates are descriptive; do not automatically promote an entire tier based on them.")


,tier,population_comments,sample_comments,reviewed_relevant,reviewed_irrelevant,reviewed_uncertain,not_yet_reviewed,sample_relevant_percent_if_n_ge_20
0,direct_comment,291858,50,45,3,2,0,93.8
1,supported_context,150882,50,39,8,3,0,83.0
2,context_only,559400,50,19,24,7,0,44.2
3,adjacent_review,89739,50,21,19,10,0,52.5
4,not_selected,550750,50,3,44,3,0,6.4


Sample rates are descriptive; do not automatically promote an entire tier based on them.
